## Cross-image patch retrieval

`05_patch_similarity_heatmap.ipynb` compared a query patch to every other patch *within the same image*. This notebook asks the cross-image version of the same question: pick a query patch in one image, then search for its nearest neighbors -- by cosine similarity of the raw patch token -- across a small gallery of *other* images (different patients, different hospitals). If the top matches show the same tissue structure as the query even though they come from a completely different slide, that's informal evidence of the same emergent part-level correspondence DINO-family ViTs are known for -- not a rigorous retrieval benchmark, just a qualitative probe.

In [ ]:
# Install dependencies, then restart the kernel. This is only needed once per environment.
# !pip install -U -r /home/shared/helper/requirements.txt

In [ ]:
import sys

sys.path.append("/home/shared/helper/")
from nbhelper import (
    plt,
    pd,
    np,
)
import os

os.environ["HF_HOME"] = "/home/shared/.cache/huggingface"
from pathlib import Path

import cv2
import openslide
import torch
from PIL import Image

from patch_similarity import denormalize, get_normalize_stats, patch_crop  # noqa: E402
from vfm_encoders import embed_image_patches, load_uni2, load_virchow2  # noqa: E402

### 0. Setup

Both encoders are gated on Hugging Face -- request access on the [UNI2-h](https://huggingface.co/MahmoodLab/UNI2-h) and [Virchow2](https://huggingface.co/paige-ai/Virchow2) model pages, then set `HF_TOKEN` (or leave unset for an interactive login prompt).

In [ ]:
import os
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN"))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print(
        "WARNING: no GPU found -- these are large ViT-H encoders, CPU is only "
        "fine for a quick smoke test on a handful of patches."
    )

### 1. Load the encoders

In [ ]:
encoders = {
    "uni2-h": load_uni2(device),
    "virchow2": load_virchow2(device),
}

# Workaround: some deployments of the shared vfm_encoders.py predate the
# Encoder.patch_tokens field this notebook needs (embed_image_patches calls
# it to get spatial tokens instead of the pooled vector). Only patches it in
# if it's actually missing, so this becomes a no-op once the shared copy is
# updated -- see this repo's own src/vfm_encoders.py for the canonical version.
if not hasattr(encoders["uni2-h"], "patch_tokens"):

    def _uni2_patch_tokens(batch, model=encoders["uni2-h"].model):
        # no_embed_class + reg_tokens=8 -> 1 CLS + 8 register tokens prefix
        return model.forward_features(batch)[:, 9:]

    encoders["uni2-h"].patch_tokens = _uni2_patch_tokens

if not hasattr(encoders["virchow2"], "patch_tokens"):

    def _virchow2_patch_tokens(batch, model=encoders["virchow2"].model):
        # model(batch) already returns the full sequence; 1 CLS + 4 register tokens prefix
        return model(batch)[:, 5:]

    encoders["virchow2"].patch_tokens = _virchow2_patch_tokens

### 2. Build a small image gallery

Two sources, mixed into one gallery:

- **CAMELYON17**: one pre-cut 96x96px patch per (center, tumor) combination -- up to 10 images spanning all 5 hospitals and both classes, so a cross-image match has to cross both a patient and (often) a stain/scanner boundary to count.
- **PANDA**: a handful of 448x448px foreground-tissue crops (native level-0 resolution, much bigger field of view than a CAMELYON patch) pulled straight from raw WSIs, one per (data provider, grade bucket) combination -- so a match can also cross datasets entirely, not just hospitals within CAMELYON17.

In [ ]:
camelyon_base_dir = Path("/home/shared/data/camelyon17/camelyon17_v1.0/")

metadata_df = (
    pd.read_csv(camelyon_base_dir / "metadata.csv", index_col=False)
    .drop("Unnamed: 0", axis=1)
    .assign(
        patient_node=lambda df_: df_.apply(
            lambda row: f"patient_{row['patient']:03d}_node_{row['node']}", axis=1
        )
    )
    .assign(
        filepath_abs=lambda df_: df_.apply(
            lambda row: camelyon_base_dir
            / "patches"
            / row["patient_node"]
            / f"patch_{row['patient_node']}_x_{row['x_coord']}_y_{row['y_coord']}.png",
            axis=1,
        )
    )
)

gallery_rows = (
    metadata_df.groupby(["center", "tumor"]).sample(1, random_state=3).reset_index(drop=True)
)
gallery_images = [Image.open(p).convert("RGB") for p in gallery_rows["filepath_abs"]]
gallery_titles = [f"center {row.center}, tumor={row.tumor}" for row in gallery_rows.itertuples()]

fig, axs = plt.subplots(
    2, (len(gallery_images) + 1) // 2, figsize=(3 * len(gallery_images) // 2, 6)
)
for ax, img, title in zip(axs.flat, gallery_images, gallery_titles):
    ax.imshow(img)
    ax.set_title(title, fontsize=8)
    ax.axis("off")
for ax in axs.flat[len(gallery_images) :]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
panda_base_dir = Path("/home/shared/data/panda/")
panda_images_dir = panda_base_dir / "train_images"

panda_df = (
    pd.read_csv(panda_base_dir / "train.csv")
    .assign(filepath=lambda df_: df_["image_id"].apply(lambda y: panda_images_dir / f"{y}.tiff"))
    .loc[lambda df_: df_["filepath"].apply(lambda p: p.exists())]
)


def sample_foreground_region(
    slide_path, size=448, foreground_threshold=0.5, thumbnail_size=1024, rng=None
):
    """Picks a random size x size tile with >= foreground_threshold Otsu tissue
    coverage and reads it at level-0 (native) resolution -- same foreground-tiling
    approach as 00_preparation/01_encodings/encode_panda.ipynb, but for picking a
    single example region rather than exhaustively tiling the whole slide."""
    rng = rng or np.random.default_rng()
    slide = openslide.OpenSlide(str(slide_path))
    try:
        width, height = slide.dimensions
        thumb_arr = np.array(slide.get_thumbnail((thumbnail_size, thumbnail_size)).convert("L"))
        _, mask = cv2.threshold(thumb_arr, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8)) > 0
        scale_x, scale_y = width / thumb_arr.shape[1], height / thumb_arr.shape[0]

        candidates = []
        for y0 in range(0, height - size + 1, size):
            my0, my1 = int(y0 / scale_y), max(int(y0 / scale_y) + 1, int((y0 + size) / scale_y))
            for x0 in range(0, width - size + 1, size):
                mx0 = int(x0 / scale_x)
                mx1 = max(mx0 + 1, int((x0 + size) / scale_x))
                if mask[my0:my1, mx0:mx1].mean() >= foreground_threshold:
                    candidates.append((x0, y0))
        if not candidates:
            raise ValueError(
                f"No foreground tile >= {foreground_threshold} coverage in {slide_path}"
            )
        x0, y0 = candidates[rng.integers(len(candidates))]
        return slide.read_region((x0, y0), level=0, size=(size, size)).convert("RGB")
    finally:
        slide.close()


panda_gallery_rows = (
    panda_df.assign(
        grade_bucket=lambda df_: np.where(
            df_["isup_grade"] >= 4, "high-grade", np.where(df_["isup_grade"] == 0, "benign", None)
        )
    )
    .dropna(subset=["grade_bucket"])
    .groupby(["data_provider", "grade_bucket"])
    .sample(1, random_state=3)
    .reset_index(drop=True)
)

panda_rng = np.random.default_rng(3)
panda_gallery_images = [
    sample_foreground_region(row.filepath, rng=panda_rng) for row in panda_gallery_rows.itertuples()
]
panda_gallery_titles = [
    f"PANDA {row.data_provider}, ISUP {row.isup_grade}" for row in panda_gallery_rows.itertuples()
]

gallery_images += panda_gallery_images
gallery_titles += panda_gallery_titles

fig, axs = plt.subplots(
    2, (len(gallery_images) + 1) // 2, figsize=(3 * len(gallery_images) // 2, 6)
)
for ax, img, title in zip(axs.flat, gallery_images, gallery_titles):
    ax.imshow(img)
    ax.set_title(title, fontsize=8)
    ax.axis("off")
for ax in axs.flat[len(gallery_images) :]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3. Token bank + cross-image nearest neighbors

Embed every gallery image into its patch-token grid, flatten all of them into one bank, then for a chosen query `(image_idx, row, col)`, rank every *other* image's patches by cosine similarity to that query token.

In [ ]:
def build_patch_bank(images, encoder, device):
    """Returns (tokens, image_idx, rows, cols, grids): tokens is every patch
    from every image flattened into one (N, D) array, with parallel arrays
    recording which image/grid-position each row came from."""
    grids = [embed_image_patches(encoder, img, device=device) for img in images]
    tokens, image_idx, rows, cols = [], [], [], []
    for i, grid in enumerate(grids):
        H, W, D = grid.shape
        tokens.append(grid.reshape(-1, D))
        rr, cc = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
        image_idx.append(np.full(H * W, i))
        rows.append(rr.reshape(-1))
        cols.append(cc.reshape(-1))
    return (
        np.concatenate(tokens),
        np.concatenate(image_idx),
        np.concatenate(rows),
        np.concatenate(cols),
        grids,
    )


def top_k_cross_image_matches(bank, query_image_idx, query_rc, k=6):
    tokens, image_idx, rows, cols, grids = bank
    query_vec = grids[query_image_idx][query_rc]
    query_n = query_vec / (np.linalg.norm(query_vec) + 1e-8)
    tokens_n = tokens / (np.linalg.norm(tokens, axis=1, keepdims=True) + 1e-8)
    sims = tokens_n @ query_n
    sims[image_idx == query_image_idx] = -np.inf  # force genuinely cross-image matches
    top_idx = np.argsort(sims)[::-1][:k]
    return [(image_idx[i], rows[i], cols[i], sims[i]) for i in top_idx]

### 4. Visualize: query patch vs. its cross-image matches

Left: the query image with the query patch boxed. Right: the top-k matching patches, cropped straight out of their source images, each labeled with similarity and where it came from.

Two example queries below: one from CAMELYON17 (as before) and one from a PANDA region, so retrieval runs in both directions across the two datasets -- note that a query's top-k matches aren't guaranteed to include the *other* dataset (lymph node and prostate tissue architecture genuinely differ), so seeing an all-CAMELYON17 or all-PANDA top-k for a given query is itself informative, not a bug.

In [ ]:
def show_cross_image_retrieval(images, titles, encoder, device, query_image_idx, query_rc, k=6):
    bank = build_patch_bank(images, encoder, device)
    matches = top_k_cross_image_matches(bank, query_image_idx, query_rc, k=k)

    mean, std = get_normalize_stats(encoder.transform)
    patch_size = encoder.model.patch_embed.patch_size[0]
    displays = [denormalize(encoder.transform(img), mean, std) for img in images]

    fig, axs = plt.subplots(1, 1 + k, figsize=(3 * (1 + k), 3.5))

    query_display = displays[query_image_idx]
    ax = axs[0]
    ax.imshow(query_display)
    r0, c0 = query_rc[0] * patch_size, query_rc[1] * patch_size
    ax.add_patch(
        plt.Rectangle(
            (c0, r0), patch_size, patch_size, edgecolor="red", facecolor="none", linewidth=2
        )
    )
    ax.set_title(f"query: {titles[query_image_idx]}", fontsize=8)
    ax.axis("off")

    for ax, (img_idx, row, col, sim) in zip(axs[1:], matches):
        crop = patch_crop(displays[img_idx], row, col, patch_size)
        ax.imshow(crop)
        ax.set_title(f"{titles[img_idx]}\nsim={sim:.2f}", fontsize=7)
        ax.axis("off")

    fig.suptitle(f"{encoder.name} -- cross-image nearest neighbors", y=1.05)
    plt.tight_layout()
    return fig


# One query from CAMELYON17, one from a PANDA region (PANDA images were appended to
# the end of the gallery in section 2, in (data_provider, grade_bucket) order --
# index -3 is the karolinska high-grade slide) -- so the retrieval demo runs both
# directions across the two datasets.
QUERIES = [
    (0, (8, 9)),
    (len(gallery_images) - 3, (8, 8)),
]

for model_name, encoder in encoders.items():
    for query_image_idx, query_rc in QUERIES:
        show_cross_image_retrieval(
            gallery_images, gallery_titles, encoder, device, query_image_idx, query_rc, k=6
        )
        plt.show()

### Takeaways

- If the retrieved crops share the query's tissue type (tumor cell nests matching other tumor regions, stroma matching stroma) despite coming from different patients/centers, the patch tokens encode something closer to *tissue morphology* than to low-level stain color -- the stronger and more useful result.
- If matches instead track color/stain intensity more than structure, that's a sign of the same acquisition-site shortcut probed for directly in `04_domain_origin_probing.ipynb`.
- With PANDA crops mixed into the gallery, a query from one dataset can now also match into the other -- a CAMELYON17 stroma patch retrieving a PANDA stroma region (or vice versa) is a stronger correspondence signal than a within-CAMELYON17 match, since it crosses tissue site (lymph node vs. prostate), scanner, and stain protocol all at once.
- Try a query patch from a visually distinctive region (a tumor nest edge, a vessel, a fat vacuole) -- ambiguous background patches tend to produce less interpretable matches for any encoder.
- This is a qualitative probe on a handful of images, not a retrieval benchmark -- treat it as a way to *look at* what a patch token represents, not to score encoders against each other.